# Scrape other website tamplate

In [ ]:
# =========================
# Extend scraping to other JobSpy-supported sites
# =========================
# This project’s main dataset uses LinkedIn for consistency.
# If you want to extend the dataset, enable the flag below and select additional sites.
# Supported sites (per JobSpy docs): linkedin, indeed, glassdoor, google, zip_recruiter, bayt, bdjobs

RUN_OTHER_SITES = False

# Choose one or more:
OTHER_SITES = ["indeed", "glassdoor", "zip_recruiter", "google"]  # edit as needed

# Optional filters (safe defaults)
LOCATION = None            # e.g., "United States" or "San Francisco, CA"
DISTANCE_MILES = 50
JOB_TYPE = None            # "fulltime", "parttime", "internship", "contract"
IS_REMOTE = None           # True/False
RESULTS_WANTED_PER_SITE = 50
HOURS_OLD = None           # e.g., 72 for last 3 days
OFFSET = 0
DESCRIPTION_FORMAT = "markdown"  # "markdown" or "html"
VERBOSE = 1                # 0 errors, 1 errors+warnings, 2 all logs

# Google Jobs uses a dedicated parameter for filtering:
GOOGLE_SEARCH_TERM = None  # e.g., "data scientist remote" (only used if "google" in OTHER_SITES)

# LinkedIn specific extras (only used if "linkedin" in OTHER_SITES)
LINKEDIN_FETCH_DESCRIPTION = True
LINKEDIN_COMPANY_IDS = None  # e.g., [123456, 7891011]

# Indeed/Glassdoor country filter (spelling must match JobSpy docs)
COUNTRY_INDEED = None       # e.g., "USA" or "United Kingdom" (use exact allowed values)

# Optional advanced options
EASY_APPLY = None           # True/False (note: LinkedIn easy apply no longer works per docs)
USER_AGENT = None
PROXIES = None              # e.g., ['user:pass@host:port', 'localhost']
ENFORCE_ANNUAL_SALARY = None
CA_CERT = None              # path to CA cert file if required for proxies

if not RUN_OTHER_SITES:
    print("Other sites template is disabled. Set RUN_OTHER_SITES = True to run it.")
else:
    # Build kwargs based on selected sites and provided filters
    scrape_kwargs_other = {
        "site_name": OTHER_SITES,
        "search_term": SEARCH_TERM,  # reuse your main SEARCH_TERM unless you want to override
        "results_wanted": RESULTS_WANTED_PER_SITE,
        "distance": DISTANCE_MILES,
        "offset": OFFSET,
        "description_format": DESCRIPTION_FORMAT,
        "verbose": VERBOSE,
    }

    # Only add optional params if they were set (keeps calls clean and compatible)
    if LOCATION:
        scrape_kwargs_other["location"] = LOCATION
    if JOB_TYPE:
        scrape_kwargs_other["job_type"] = JOB_TYPE
    if IS_REMOTE is not None:
        scrape_kwargs_other["is_remote"] = IS_REMOTE
    if HOURS_OLD is not None:
        scrape_kwargs_other["hours_old"] = HOURS_OLD
    if EASY_APPLY is not None:
        scrape_kwargs_other["easy_apply"] = EASY_APPLY
    if USER_AGENT:
        scrape_kwargs_other["user_agent"] = USER_AGENT
    if PROXIES:
        scrape_kwargs_other["proxies"] = PROXIES
    if LINKEDIN_FETCH_DESCRIPTION is not None:
        scrape_kwargs_other["linkedin_fetch_description"] = LINKEDIN_FETCH_DESCRIPTION
    if LINKEDIN_COMPANY_IDS:
        scrape_kwargs_other["linkedin_company_ids"] = LINKEDIN_COMPANY_IDS
    if COUNTRY_INDEED:
        scrape_kwargs_other["country_indeed"] = COUNTRY_INDEED
    if ENFORCE_ANNUAL_SALARY is not None:
        scrape_kwargs_other["enforce_annual_salary"] = ENFORCE_ANNUAL_SALARY
    if CA_CERT:
        scrape_kwargs_other["ca_cert"] = CA_CERT

    # Google jobs: only param for filtering is google_search_term
    # If user includes "google" and provides GOOGLE_SEARCH_TERM, pass it in.
    if "google" in [s.lower() for s in OTHER_SITES] and GOOGLE_SEARCH_TERM:
        scrape_kwargs_other["google_search_term"] = GOOGLE_SEARCH_TERM

    # Run scrape
    df_other_raw = scrape_jobs(**scrape_kwargs_other)

    # Standardize/clean using your existing helper
    df_other_clean = standardize_columns(df_other_raw)

    # Save outputs (timestamped)
    other_sites_tag = "-".join([s.lower() for s in OTHER_SITES])
    other_raw_path = os.path.join(DATA_RAW_DIR, f"jobs_{other_sites_tag}_{RUN_TS}.csv")
    other_clean_path = os.path.join(DATA_CLEAN_DIR, f"jobs_{other_sites_tag}_cleaned_{RUN_TS}.csv")

    df_other_raw.to_csv(other_raw_path, index=False)
    df_other_clean.to_csv(other_clean_path, index=False)

    print("Saved other sites raw to:", other_raw_path)
    print("Saved other sites cleaned to:", other_clean_path)
    print("Rows collected:", len(df_other_clean))